<a href="https://colab.research.google.com/github/SugiShanmu/Machine-Learning-/blob/main/SampleDataProcessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import re
import os

# 1. CREATE SAMPLE INPUT (if you don't have file)
sample_data = {
    'ID': [1,2,3,4,5,6,7,8,9,10,11],
    'Name': ['Ravi Kumar','Anita S','Ravi Kumar','','John Doe','Priya','Karan',' sara ali ','Amit','Vijay','Amit'],
    'Email': ['ravi@gmail.com','ANITA@GMAIL.COM','ravi@gmail.com','test@gmail.com','invalid-email','priya@yahoo.com','karan@gmail.com','sara.ali@outlook.com','amit@gmail.com','vijay@gmail.com','amit@gmail.com'],
    'Age': [25,30,25,22,28,'',-5,35,40,29,40],
    'City': ['Chennai','Bangalore','Chennai','Delhi','Mumbai','Hyderabad','Pune',' chennai ','Delhi','Chennai','Delhi'],
    'Amount': [1000,1500,1000,800,2000,1200,500,3000,'',2500,'']
}
df = pd.DataFrame(sample_data)
df.to_csv("sample_input.csv", index=False)
print("Sample created: sample_input.csv")

# 2. CLEANING FUNCTION
def validate_email(email):
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, str(email).strip()))

# Load
try:
    df = pd.read_csv("sample_input.csv")
except:
    df = pd.read_csv("/content/sample_input.csv")

print(f"Initial records: {len(df)}")

# Clean spaces & lower
df.columns = [c.strip().lower() for c in df.columns]
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'nan': None, 'None': None, '': None, ' ': None, 'nan': None})

if 'name' in df.columns:
    df['name'] = df['name'].apply(lambda x: str(x).title() if x and pd.notna(x) else x)
if 'email' in df.columns:
    df['email'] = df['email'].astype(str).str.lower()
if 'city' in df.columns:
    df['city'] = df['city'].apply(lambda x: str(x).title() if x and pd.notna(x) else x)

# Remove duplicates
before = len(df)
df = df.drop_duplicates()
if 'email' in df.columns:
    df = df.drop_duplicates(subset=['email'], keep='first')
print(f"Duplicates removed: {before - len(df)}")

# Find invalid
invalid_mask = pd.Series(False, index=df.index)
if 'email' in df.columns:
    invalid_mask |= ~df['email'].apply(lambda x: validate_email(x) if pd.notna(x) and x not in [None,'none'] else False)
if 'age' in df.columns:
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    invalid_mask |= df['age'].isna() | (df['age'] < 0) | (df['age'] > 100)
for c in ['name','email']:
    if c in df.columns:
        invalid_mask |= df[c].isna() | (df[c].astype(str).str.lower() == 'none')

print(f"Invalid found: {invalid_mask.sum()}")

df_valid = df[~invalid_mask].copy()
df_invalid = df[invalid_mask].copy()

# Filter & Sort
if 'amount' in df_valid.columns:
    df_valid['amount'] = pd.to_numeric(df_valid['amount'], errors='coerce')
    df_valid = df_valid[df_valid['amount'] > 0]
if 'city' in df_valid.columns:
    df_valid = df_valid.sort_values(by=['city','name'])

print(f"Final valid: {len(df_valid)}")

# 3. SAVE - This generates files in Colab
df_valid.to_excel("cleaned_output.xlsx", index=False)
df_invalid.to_csv("invalid_records.csv", index=False)

# Create report
report = f"""
DATA PROCESSING REPORT

Initial: {len(df) + (before - len(df))}
Duplicates: {before - len(df)}
Invalid: {invalid_mask.sum()}
Valid Final: {len(df_valid)}
"""
print(report)
with open("processing_report.txt", "w") as f:
    f.write(report)

print("\nDONE! Files generated:")
print("1. cleaned_output.xlsx")
print("2. invalid_records.csv")
print("3. processing_report.txt")

Sample created: sample_input.csv
Initial records: 11
Duplicates removed: 2
Invalid found: 4
Final valid: 4

DATA PROCESSING REPORT

Initial: 11
Duplicates: 2
Invalid: 4
Valid Final: 4


DONE! Files generated:
1. cleaned_output.xlsx
2. invalid_records.csv
3. processing_report.txt
